# Taller 1 — Programación Dinámica
## Milan Taxi (Versión en Español)

**Curso:** Aprendizaje por Refuerzo  |  **Fecha:** Septiembre 2026

**Objetivo:** Implementar y analizar PI y VI en el entorno MilanTaxi.

## 1. Objetivo

Implementar dos algoritmos de PD (Sutton & Barto Cap. 4):

- **Iteración de Políticas:** Alterna evaluación y mejora.
- **Iteración de Valores:** Computa $V^*$ via el operador de Bellman de optimalidad.

Aplicamos a **MilanTaxi** (cuadrícula 5x5, 500 estados) en dos variantes:
- **Original:** Transiciones deterministas.
- **Estocástica:** Deslizamiento 0.1 en acciones de movimiento.

## 2. Formulación del MDP

Un MDP es $(S, A, P, R, gamma)$.

**Espacio de Estados:** $s$ = (fila, col, pasajero, destino)

| Componente | Rango | Descripción |
|---|---|---|
| Fila | [0,4] | Fila del taxi |
| Columna | [0,4] | Columna del taxi |
| Pasajero | [0,4] | 0=no recogido, 1-4=ubicación |
| Destino | [0,3] | Destino del pasajero |

**Codificación:** fila*100 + col*20 + pasajero*4 + destino

**Acciones:** SUR, NORTE, ESTE, OESTE, RECOGER, DEJAR |A|=6

**Recompensas:** -1 por paso, -10 invalido, +20 DEJAR exitoso

**gamma=0.99**, **Horizonte:** 100 pasos (siempre trunca).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time, sys, os
sys.path.insert(0, os.path.abspath("../src"))
from rl_project.envs.milan_taxi import MilanTaxiEnv,N_S,N_A,HORIZON,encode_state,decode_state,SOUTH,NORTH,EAST,WEST,PICKUP,DROPOFF,LOCS
from rl_project.models.mdp import build_model,check_probability_distribution
from rl_project.agents.dynamic_programming import policy_iteration,value_iteration
from rl_project.evaluation import evaluate_policy
from rl_project import policies
sns.set_theme(style="whitegrid")
%matplotlib inline

GAMMA=0.99
TOL=1e-10
S0=encode_state(0,0,0,1)

## 3. Entorno — Variante Original

Transiciones deterministas.

In [ ]:
env_orig=MilanTaxiEnv(variant="original")
print("Variante:",env_orig.variant,"|S|=",N_S,"|A|=",N_A,"Horizonte=",HORIZON)
state,_=env_orig.reset(seed=42)
print("Estado inicial:",state,"->",decode_state(encode_state(*state)))
print(env_orig.render())
for name,a in [("SUR",0),("ESTE",2),("ESTE",2)]:
    ns,r,_,_,_=env_orig.step(a)
    print(f"{name}: sig={ns}, r={r}")

## 4. Entorno — Variante Estocastica

Deslizamiento: P(intencionado)=0.8, P(izq)=0.1, P(der)=0.1. RECOGER/DEJAR deterministas.

In [ ]:
env_stoch=MilanTaxiEnv(variant="stochastic",slip_probability=0.1)
print("Slip probability:",env_stoch.slip_probability)
print("\n=== Mov. estocastico (ESTE x10) ===")
for t in range(10):
    env_stoch.reset(seed=0)
    ns,r,_,_,_=env_stoch.step(EAST)
    print(f"  Intento {t+1}: sig={ns} -> {decode_state(encode_state(*ns))}")

## 5. Modelo MDP — Construccion de $P$ y $R$

`build_model()` construye $P(s,a,s')$ y $R(s,a)$.

In [ ]:
P_orig,R_orig=build_model(MilanTaxiEnv(variant="original"))
P_stoch,R_stoch=build_model(MilanTaxiEnv(variant="stochastic",slip_probability=0.1))
print("P_orig:",P_orig.shape,"| R_orig:",R_orig.shape)
print("P_stoch:",P_stoch.shape,"| R_stoch:",R_stoch.shape)
assert P_orig.shape==(N_S,N_A,N_S) and R_orig.shape==(N_S,N_A)
assert check_probability_distribution(P_orig) and check_probability_distribution(P_stoch)
assert (P_orig>=0).all() and (P_stoch>=0).all()
assert (P_orig.sum(axis=-1)-1.0<1e-10).all()
print("Validacion P/R: OK")

### 5.1 Original vs Estocasticas

Original: 1 sucesor. Estocastico: hasta 3 sucesores para movimiento.

In [ ]:
s_test=encode_state(2,2,1,3)
print(f"s={decode_state(s_test)}")
a_test=EAST
print(f"a={a_test} (ESTE)")
nz_orig=np.count_nonzero(P_orig[s_test,a_test])
nz_stoch=np.count_nonzero(P_stoch[s_test,a_test])
print(f"  Original: {nz_orig} sucesor(es)")
print(f"  Estoc.: {nz_stoch} sucesor(es)")
print(f"  P_orig={P_orig[s_test,a_test]}")
print(f"  P_stoch={P_stoch[s_test,a_test]}")

## 6. Iteracion de Politicas (PI)

**Algoritmo:** 1) pi0 aleatoria. 2) Evaluar: (I-gamma P^pi) V^pi = R^pi. 3) Mejorar: greedy. 4) Repetir.

In [ ]:
print("=== PI — Original ===")
t0=time.perf_counter()
V_pi_orig,pi_pi_orig,n_pi_orig=policy_iteration(P_orig,R_orig,GAMMA)
t_pi_orig=time.perf_counter()-t0
print(f"  Iter: {n_pi_orig} | V*(0)={V_pi_orig[S0]:.4f} | T={t_pi_orig:.3f}s")
print("\n=== PI — Estocastico ===")
t0=time.perf_counter()
V_pi_stoch,pi_pi_stoch,n_pi_stoch=policy_iteration(P_stoch,R_stoch,GAMMA)
t_pi_stoch=time.perf_counter()-t0
print(f"  Iter: {n_pi_stoch} | V*(0)={V_pi_stoch[S0]:.4f} | T={t_pi_stoch:.3f}s")

### Observaciones PI

- Converge en 7-11 iteraciones.
- Cada iteracion resuelve un sistema 500x500.
- Determinista requiere mas iteraciones (11 vs 7).

## 7. Iteracion de Valores (VI)

Algoritmo:

V_{k+1}(s) = max_a [ R(s,a) + gamma * sum_{s"} P(s"|s,a) V_k(s") ]

Luego extraer politica greedy.

In [ ]:
print("=== VI — Original ===")
t0=time.perf_counter()
V_vi_orig,pi_vi_orig,n_sweeps_orig,deltas_orig=value_iteration(P_orig,R_orig,GAMMA,tol=TOL,max_iter=100000)
t_vi_orig=time.perf_counter()-t0
print(f"  Barridos: {n_sweeps_orig} | V*(0)={V_vi_orig[S0]:.4f} | T={t_vi_orig:.3f}s | df={deltas_orig[-1]:.2e}")
print("\n=== VI — Estocastico ===")
t0=time.perf_counter()
V_vi_stoch,pi_vi_stoch,n_sweeps_stoch,deltas_stoch=value_iteration(P_stoch,R_stoch,GAMMA,tol=TOL,max_iter=100000)
t_vi_stoch=time.perf_counter()-t0
print(f"  Barridos: {n_sweeps_stoch} | V*(0)={V_vi_stoch[S0]:.4f} | T={t_vi_stoch:.3f}s | df={deltas_stoch[-1]:.2e}")

### Observaciones VI

- ~2600 barridos con gamma=0.99, tol=1e-10.
- Cada barrido: O(|S|^2 * |A|).
- Tiempo total ~6s comparable a PI.

## 8. Convergencia de VI

Delta maximo max_s |V_{k+1} - V_k| con los barridos.

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
ax.semilogy(deltas_orig,label="Original",linewidth=2)
ax.semilogy(deltas_stoch,label="Estocastico",linewidth=2,linestyle="--")
ax.axhline(TOL,color="gray",linestyle=":",label=f"tol={TOL}")
ax.set_xlabel("Barrido")
ax.set_ylabel("Delta maximo")
ax.set_title("Convergencia de Value Iteration")
ax.legend()
ax.grid(True,which="both",alpha=0.3)
plt.tight_layout()
from pathlib import Path
cwd = Path.cwd()
if (cwd / "runs").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "runs").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta runs/ desde el directorio actual."
    )
RUNS_DIR = PROJECT_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
VI_CONVERGENCE_PATH = RUNS_DIR / "vi_convergence.png"
fig.savefig(VI_CONVERGENCE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardada en: {VI_CONVERGENCE_PATH}")


## 9. Visualización iterativa de Value Iteration

Las siguientes animaciones muestran cómo evoluciona la política durante
las iteraciones de **Value Iteration** (barridos) y cómo se comporta
el taxi bajo dicha política en el entorno MilanTaxi.

Cada animación captura *snapshots* de la política $\pi_k$ obtenida
en distintas iteraciones $ del algoritmo. Para cada snapshot:

1. Se extrae la política greedy $\pi_k(s) = \arg\max_a Q_k(s,a)$.
2. Se inicia el taxi desde un estado reproducible.
3. Se ejecuta la política por algunos pasos.
4. Se visualiza: taxi, pasajero, destino, acción, recompensa.

Esto permite **vincular conceptualmente** la evolución del algoritmo
con el comportamiento observable del agente.


In [ ]:
from IPython.display import HTML, display
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
from rl_project.agents.dynamic_programming import q_from_v, greedy_policy
from rl_project.envs.milan_taxi import (
    MilanTaxiEnv, N_S, N_A, HORIZON,
    encode_state, decode_state,
    SOUTH, NORTH, EAST, WEST, PICKUP, DROPOFF, LOCS, INTERNAL_WALLS,
    MOVE_DELTAS, _SLIP_MAP, check_wall, _compute_new_position,
)
from rl_project.models.mdp import build_model

ACTION_NAMES = {SOUTH: "SUR", NORTH: "NORTE", EAST: "ESTE", WEST: "OESTE",
                PICKUP: "RECOGER", DROPOFF: "DEJAR"}
PASSENGER_LOCATION_NAMES = {0: "R(0,0)", 1: "G(0,4)", 2: "Y(4,0)", 3: "B(4,3)", 4: "En taxi"}
DESTINATION_NAMES = {0: "R(0,0)", 1: "G(0,4)", 2: "Y(4,0)", 3: "B(4,3)"}
COLORS = {0: "red", 1: "green", 2: "gold", 3: "blue"}

def draw_milan_taxi_state(
    ax, state, action_idx=None, reward=None,
    vi_iter=None, step=None, variant="original",
    delta=None, pi_action_idx=None, actual_action_idx=None,
):
    ax.clear()
    row, col, pass_idx, dest_idx = state
    for r in range(5):
        for c in range(5):
            rect = mpatches.Rectangle((c, 4-r), 1, 1,
                                      linewidth=0.5, edgecolor="gray",
                                      facecolor="white", zorder=1)
            ax.add_patch(rect)
    for (r1,c1),(r2,c2) in INTERNAL_WALLS:
        y1,x1=4-r1,c1; y2,x2=4-r2,c2
        if r1==r2:
            ax.plot([x1+1,x2],[y1+0.5,y2+0.5],color="black",linewidth=4,zorder=5)
        elif c1==c2:
            ax.plot([x1+0.5,x2+0.5],[y1,y2+1],color="black",linewidth=4,zorder=5)
        else:
            ax.plot([x1+0.5,x2+0.5],[y1+0.5,y2+0.5],color="black",linewidth=4,zorder=5)
    dr,dc=LOCS[dest_idx]
    ax.add_patch(mpatches.Circle((dc+0.5,4-dr+0.5),0.35,color=COLORS[dest_idx],alpha=0.25,zorder=2))
    ax.text(dc+0.5,4-dr+0.5,"\U0001f3af",fontsize=16,ha="center",va="center",zorder=6)
    if pass_idx!=4:
        pr,pc=LOCS[pass_idx]
        ax.add_patch(mpatches.Circle((pc+0.5,4-pr+0.5),0.35,color=COLORS[pass_idx],alpha=0.15,zorder=2))
        ax.text(pc+0.5,4-pr+0.5,"\U0001f9cd",fontsize=16,ha="center",va="center",zorder=6)
    tc="yellow" if variant=="original" else "orange"
    rect=mpatches.Rectangle((col+0.05,4-row+0.05),0.9,0.9,
                            linewidth=2,edgecolor="black",facecolor=tc,alpha=0.7,zorder=3)
    ax.add_patch(rect)
    label="\U0001f695\U0001f9d1" if pass_idx==4 else "\U0001f695"
    ax.text(col+0.5,4-row+0.5,label,fontsize=16,ha="center",va="center",zorder=6)
    if action_idx is not None and action_idx in (SOUTH,NORTH,EAST,WEST):
        dr,dc=MOVE_DELTAS[action_idx]
        ax.arrow(col+0.5,4-row+0.5,dc*0.35,-dr*0.35,
                 head_width=0.2,head_length=0.15,fc="red",ec="red",linewidth=2,zorder=7)
    info=[]
    if vi_iter is not None: info.append("VI Iteraci\u00f3n: {}".format(vi_iter))
    if step is not None: info.append("Paso del episodio: {}".format(step))
    info.append("Taxi: ({},{})".format(row,col))
    info.append("Pasajero: {}".format(PASSENGER_LOCATION_NAMES[pass_idx]))
    info.append("Destino: {}".format(DESTINATION_NAMES[dest_idx]))
    if action_idx is not None:
        an=ACTION_NAMES.get(action_idx,"?")
        if pi_action_idx is not None and pi_action_idx!=action_idx:
            pn=ACTION_NAMES.get(pi_action_idx,"?")
            info.append("Acci\u00f3n pol\u00edtica: {}".format(pn))
            info.append("Movimiento real: {} \u26a0".format(an))
        else:
            info.append("Acci\u00f3n: {}".format(an))
    if reward is not None: info.append("Reward: {:+d}".format(int(reward)))
    if delta is not None: info.append("Delta VI: {:.2e}".format(delta))
    ax.text(5.5,3.5,"\n".join(info),fontsize=9,verticalalignment="top",
            fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.3",facecolor="lightyellow",edgecolor="gray",alpha=0.9))
    ax.set_xlim(0,8);ax.set_ylim(0,5);ax.set_aspect("equal");ax.axis("off")
    ax.set_title("Value Iteration \u2014 {}".format(
        "Determinista" if variant=="original" else "Estoc\u00e1stico"),
        fontsize=12,fontweight="bold")

def value_iteration_snapshots(P,R,gamma,tol=1e-10,max_iter=100000):
    V=np.zeros(R.shape[0]);deltas_list=[];snapshots=[]
    Q0=q_from_v(P,R,V,gamma);pi0=greedy_policy(Q0)
    snapshots.append((0,V.copy(),0.0,pi0))
    for k in range(1,max_iter+1):
        Q=q_from_v(P,R,V,gamma);V_new=Q.max(axis=1)
        delta=float(np.max(np.abs(V_new-V)))
        deltas_list.append(delta);V=V_new
        ci={1,2,3,4,5,10,20,50,100,200,500,1000,1500,2000,2500}
        if k in ci or (k>0 and k%500==0):
            pik=greedy_policy(q_from_v(P,R,V,gamma))
            snapshots.append((k,V.copy(),delta,pik))
        if delta<tol: break
    Qf=q_from_v(P,R,V,gamma);pif=greedy_policy(Qf)
    if snapshots[-1][0]!=k:
        snapshots.append((k,V.copy(),deltas_list[-1]if deltas_list else 0.0,pif))
    print("  VI: {} barridos, {} snapshots".format(k,len(snapshots)))
    return V,pif,k,deltas_list,snapshots

def animate_vi_milan_taxi(variant="original",slip_probability=0.1,seed=42,
                           max_episode_steps=8,n_episode_steps=5,
                           interval=1200,tol=1e-10):
    if variant=="original":
        env=MilanTaxiEnv(variant="original")
    else:
        env=MilanTaxiEnv(variant="stochastic",slip_probability=slip_probability)
    P,R=build_model(env);gamma=GAMMA
    print("VI snapshots -- Variante: {}".format(variant))
    Vf,pif,nsweeps,deltas,snaps=value_iteration_snapshots(P,R,gamma,tol=tol)
    rng=np.random.default_rng(seed)
    env.reset(seed=int(rng.integers(0,2**31)))
    start_state=tuple(env.unwrapped.state)
    print("  Estado inicial: {}".format(start_state))
    frames=[]
    for vi_k,Vk,dk,pik in snaps:
        if vi_k==0 and len(snaps)>3: continue
        state=start_state
        for step in range(n_episode_steps):
            sidx=encode_state(*state);act=int(pik[sidx].argmax())
            if variant=="original":
                nr,nc,np_,nd,r,_=env.get_deterministic_outcome(*state,act)
                actual=act
            else:
                outcomes=env.get_stochastic_outcomes(*state,act)
                probs=[o[0] for o in outcomes]
                idx=rng.choice(len(outcomes),p=probs)
                prob,nr,nc,np_,nd,r,_=outcomes[idx];actual=act
                for at in (SOUTH,NORTH,EAST,WEST):
                    tr,tc=_compute_new_position(state[0],state[1],at)
                    if check_wall(state[0],state[1],tr,tc): tr,tc=state[0],state[1]
                    if (tr,tc)==(nr,nc): actual=at;break
            frames.append({"vi_iter":vi_k,"step":step,"state":state,"action":act,
                           "pi_action":act,"actual_action":actual,"reward":r,
                           "delta":dk if step==0 else None,"next_state":(nr,nc,np_,nd)})
            state=(nr,nc,np_,nd)
        frames.append({"vi_iter":vi_k,"step":"-","state":state,"action":None,
                       "pi_action":None,"actual_action":None,"reward":None,
                       "delta":dk,"next_state":None,"pause":True})
    print("  Frames: {}".format(len(frames)))
    fig,ax=plt.subplots(figsize=(8,5))
    def upd(n):
        fd=frames[n];s=fd["state"]
        sw=variant=="stochastic" and fd.get("pi_action") is not None and fd.get("actual_action")!=fd.get("pi_action")
        draw_milan_taxi_state(ax,s,
            action_idx=fd.get("actual_action") if sw else fd.get("action"),
            reward=fd.get("reward"),vi_iter=fd.get("vi_iter"),step=fd.get("step"),
            variant=variant,delta=fd.get("delta"),
            pi_action_idx=fd.get("pi_action") if sw else None,
            actual_action_idx=fd.get("actual_action") if sw else None)
    anim=animation.FuncAnimation(fig,upd,frames=len(frames),interval=interval,repeat=True)
    anim._frames_data=frames;plt.close(fig)
    return anim,frames,snaps,nsweeps


In [ ]:
print("="*60)
print("9.1 — Escenario Determinista")
print("="*60)
print("\nP(s'|s,a) = 1 para el sucesor correspondiente.\n")
anim_det, frames_det, snaps_det, sw_det = animate_vi_milan_taxi(
    variant="original", seed=42,
    max_episode_steps=8, n_episode_steps=5,
    interval=1200, tol=TOL,
)


In [ ]:
print("\n▶ Reproduciendo animación determinista...")
display(HTML(anim_det.to_jshtml()))
# Guardar GIF si Pillow está disponible
try:
    from matplotlib.animation import PillowWriter
    det_gif = RUNS_DIR / "vi_milan_taxi_deterministic.gif"
    anim_det.save(det_gif, writer=PillowWriter(fps=1))
    print(f"GIF guardado en: {det_gif}")
except (ImportError, Exception) as e:
    print(f"Advertencia: No se pudo guardar GIF ({e}). La animación HTML está disponible.")


In [ ]:
print("="*60)
print("9.2 — Escenario Estocástico")
print("="*60)
print("\nP(s'|s,a) con 0.8 intención, 0.1 izquierda, 0.1 derecha.\n")
anim_stoch, frames_stoch, snaps_stoch, sw_stoch = animate_vi_milan_taxi(
    variant="stochastic", slip_probability=0.1, seed=42,
    max_episode_steps=8, n_episode_steps=5,
    interval=1500, tol=TOL,
)


In [ ]:
print("\n▶ Reproduciendo animación estocástica...")
display(HTML(anim_stoch.to_jshtml()))
# Guardar GIF si Pillow está disponible
try:
    from matplotlib.animation import PillowWriter
    stoch_gif = RUNS_DIR / "vi_milan_taxi_stochastic.gif"
    anim_stoch.save(stoch_gif, writer=PillowWriter(fps=1))
    print(f"GIF guardado en: {stoch_gif}")
except (ImportError, Exception) as e:
    print(f"Advertencia: No se pudo guardar GIF ({e}). La animación HTML está disponible.")


## 9.3 Comparación visual

### Determinista

En el escenario **determinista** (original), cada acción de movimiento
produce una transición conocida:

P(s' \mid s, a) = 1

para el sucesor correspondiente. La acción elegida por la política
se ejecuta siempre sin desviación.

### Estocástico

En el escenario **estocástico**, las acciones de movimiento tienen
una probabilidad de deslizamiento (*slip*) de 0.1:

- (\text{intención}) = 0.8$
- (\text{izquierda}) = 0.1$
- (\text{derecha}) = 0.1$

Esto significa que el taxi puede desviarse de la acción prevista.
En la animación, cuando ocurre un deslizamiento se muestra:

> **⚠️ Acción política: NORTE → Movimiento real: OESTE**

Esta diferencia es fundamental porque modifica la dinámica del MDP:
el agente ya no puede predecir exactamente el resultado de sus acciones,
lo que resulta en una política más conservadora y un valor ^*$ menor.


## 9.4 Interpretación

A medida que **Value Iteration** avanza, la política $\pi_k$ evoluciona:

1. **Iteración 0:** =0$, política esencialmente aleatoria (o constante).
   El taxi se mueve sin dirección clara.
2. **Iteraciones tempranas (1-10):** El taxi comienza a evitar paredes
   y a moverse hacia ubicaciones relevantes, pero aún sin una estrategia
   óptima.
3. **Iteraciones intermedias (10-100):** La política se vuelve más
   coherente: recoger pasajero, dirigirse al destino.
4. **Iteraciones avanzadas (100-500):** La política se estabiliza.
   Los cambios son mínimos.
5. **Iteración final:** La política óptima $\pi^*$.

La animación permite **observar** cómo la política se "endurece"
desde un comportamiento errático hasta una estrategia óptima.


## 9.5 Tabla comparativa

| Característica | Determinista | Estocástico |
|---|---|---|
| Probabilidad transición | (s' \mid s,a) = 1$ | (\text{intención})=0.8$, (\text{slip})=0.2$ |
| Slip (deslizamiento) | No | Sí (0.1 por lado) |
| Política | Óptima determinista | Óptima estocástica |
| Comportamiento taxi | Predecible | A veces se desvía |
| Valor inicial ^*(s_0)$ | ≈ 1818.39 | ≈ 1771.30 |
| Barridos VI | ≈ 2591 | ≈ 2592 |

La política estocástica produce un valor esperado menor porque
el agente no tiene control completo sobre el movimiento.
Esto se refleja en una política más cautelosa.


## 10. Comparacion PI vs VI

Verificar que producen la misma V* y politicas similares.


In [ ]:
print("=== Max diferencia V* ===")
max_diff_orig=np.max(np.abs(V_pi_orig-V_vi_orig))
max_diff_stoch=np.max(np.abs(V_pi_stoch-V_vi_stoch))
print(f"  Original: {max_diff_orig:.2e}")
print(f"  Estoc.: {max_diff_stoch:.2e}")
print("\n=== Acuerdo de politicas ===")
agr_orig=policies.policy_agreement(pi_pi_orig,pi_vi_orig)
agr_stoch=policies.policy_agreement(pi_pi_stoch,pi_vi_stoch)
print(f"  Original: {agr_orig*100:.1f}%")
print(f"  Estoc.: {agr_stoch*100:.1f}%")

## 11. Evaluacion Empirica

1000 episodios Monte Carlo (seed=42, horizonte=100).


In [ ]:
print("Evaluando (1000 eps)...")
eval_res=[]
for n,e,p in [("Orig+PI",MilanTaxiEnv(variant="original"),pi_pi_orig),("Orig+VI",MilanTaxiEnv(variant="original"),pi_vi_orig),("Stoch+PI",MilanTaxiEnv(variant="stochastic",slip=0.1),pi_pi_stoch),("Stoch+VI",MilanTaxiEnv(variant="stochastic",slip=0.1),pi_vi_stoch)]:
    r=evaluate_policy(e,p,n_runs=1000,gamma=GAMMA,seed=42,verbose=False)
    r["name"]=n
    eval_res.append(r)
print("OK")

In [ ]:
print("\n|Variante|gamma|Barridos|V*(0)|Tiempo|")
print("|-|-|-|-|-|")
for l,g,ns,v0,rt in gd:
    print(f"|{l}|{g:.2f}|{ns}|{v0:+.2f}|{rt:.3f}|")

### Obs. sobre gamma

- gamma=0.5: V* negativo (miope).
- gamma=0.9: V* positivo. 248 barridos.
- gamma=0.99: Alto V* pero ~2600 barridos.

Variante estocastica tiene V* mas bajo.

## 13. Preguntas Guia

### Q1: Que es el MDP?
(S,A,P,R,gamma) con 500 estados, 6 acciones, gamma=0.99.

### Q2: Que cambia con transiciones no-deterministas?
P(s'|s,a) pasa de delta a distribucion; V* disminuye.

### Q3: Diferencia fundamental PI vs VI?
PI = dos bucles (Newton), VI = un bucle (descenso).

### Q4: Como sabemos que convergio?
Delta < tol (VI), politica estable (PI), validacion cruzada.

### Q5: Efecto de modificacion estocastica?
V* cae 2.6%, retorno empirico cae ~80%, politica conservadora.


## 14. Limitaciones

1. Horizonte finito (100 pasos) -> brecha teorico-empirica.
2. No terminal.
3. Solo movimiento estocastico.
4. VI ~2600 barridos.
5. Empates en Q (argmax arbitrario).
6. PD tabular no escala.


## 15. Conclusiones

1. MDP critico: 500 estados, 6 acciones, rico y manejable.
2. Estocasticidad reduce retorno (~80% empirico).
3. PI y VI convergen al mismo V*.
4. Trade-off: PI pocas iter caras vs VI muchas baratas.
5. gamma bajo -> miope, gamma alto -> lento.
6. PD tabular no escala.


In [ ]:
print("="*60)
print("EJECUCION DEL NOTEBOOK COMPLETADA")
print("="*60)
print(f"PI it: Orig={n_pi_orig}, Stoch={n_pi_stoch}")
print(f"VI sw: Orig={n_sweeps_orig}, Stoch={n_sweeps_stoch}")
print(f"V*(0) PI orig={V_pi_orig[S0]:.4f}")
print(f"V*(0) VI orig={V_vi_orig[S0]:.4f}")
print(f"V*(0) PI stoch={V_pi_stoch[S0]:.4f}")
print(f"V*(0) VI stoch={V_vi_stoch[S0]:.4f}")
print(f"Acuerdo PI-VI orig={agr_orig*100:.1f}%")
print(f"Acuerdo PI-VI stoch={agr_stoch*100:.1f}%")
print("="*60)